[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week06.ipynb)

# Week 6: Optimization
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

Gradient descent; SGD, momentum, and Adam; learning rates and optimization dynamics.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Understand SGD, momentum, and Adam.
- Reason about learning rates and convergence.
- Tune an optimizer to a target.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. One model, three optimizers
SGD, SGD+momentum, and Adam on the same problem.

In [ ]:
torch.manual_seed(0)
X = torch.randn(200, 5); y = X @ torch.randn(5, 1) + 0.1 * torch.randn(200, 1)

def run(name, steps=80):
    torch.manual_seed(0); m = nn.Linear(5, 1); f = nn.MSELoss()
    o = {"SGD": torch.optim.SGD(m.parameters(), lr=0.05),
         "Momentum": torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9),
         "Adam": torch.optim.Adam(m.parameters(), lr=0.05)}[name]
    h = []
    for _ in range(steps):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    return h

for name in ["SGD", "Momentum", "Adam"]:
    plt.plot(run(name), label=name)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Optimizer comparison"); plt.show()

## 2. What momentum does, on a 2D surface
Watch SGD vs momentum descend an elongated (ill-conditioned) bowl.

In [ ]:
# loss = 0.5*(a*x^2 + y^2): steep in x, shallow in y -> SGD zig-zags
def path(momentum, steps=40, lr=0.1):
    p = torch.tensor([4.0, 4.0], requires_grad=True)
    o = torch.optim.SGD([p], lr=lr, momentum=momentum); pts = []
    for _ in range(steps):
        o.zero_grad(); loss = 0.5 * (10 * p[0] ** 2 + p[1] ** 2); loss.backward(); o.step()
        pts.append(p.detach().clone())
    return torch.stack(pts)
for mtm in [0.0, 0.9]:
    pts = path(mtm); plt.plot(pts[:, 0], pts[:, 1], "-o", ms=3, label=f"momentum={mtm}")
plt.scatter([0], [0], c="k", marker="*", s=120); plt.legend(); plt.title("Momentum smooths the path"); plt.show()

## 3. The learning rate is everything
A three-rate sweep: too small, good, too large.

In [ ]:
for lr in [0.001, 0.05, 0.5]:
    torch.manual_seed(0); m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=lr); f = nn.MSELoss()
    h = []
    for _ in range(80):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    plt.plot(h, label=f"lr={lr}")
plt.yscale("log"); plt.legend(); plt.xlabel("step"); plt.ylabel("loss (log)"); plt.show()

## 4. Read the curve to diagnose
Spikes = too large; flat = too small; smooth decline = good.

In [ ]:
torch.manual_seed(0); m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=1.2); f = nn.MSELoss()
h = []
for _ in range(40):
    o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
print("loss exploded to:", h[-1])
plt.plot(h); plt.title("lr too large: divergence"); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

## 5. Batch size changes the gradient noise
Smaller batches add useful noise to the descent path.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
ds = TensorDataset(X, y)
for bs in [8, 200]:
    torch.manual_seed(0); m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), 0.05); f = nn.MSELoss(); h = []
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    for _ in range(15):
        for xb, yb in dl:
            o.zero_grad(); l = f(m(xb), yb); l.backward(); o.step(); h.append(l.item())
    plt.plot(h, label=f"batch={bs}")
plt.legend(); plt.xlabel("update"); plt.ylabel("loss"); plt.title("Small batch = noisier updates"); plt.show()

## 6. A learning-rate schedule
Step decay: lower the rate as training settles.

In [ ]:
m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=0.2); f = nn.MSELoss()
sched = torch.optim.lr_scheduler.StepLR(o, step_size=25, gamma=0.3)
lrs, h = [], []
for _ in range(80):
    o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); sched.step()
    lrs.append(o.param_groups[0]["lr"]); h.append(l.item())
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(lrs); ax[0].set_title("learning rate"); ax[1].plot(h); ax[1].set_title("loss")
plt.show()

**Try it live:** swap `StepLR` for `CosineAnnealingLR(o, T_max=80)` and compare the curves.

In [ ]:
m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=0.2); f = nn.MSELoss()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=80)
lrs = []
for _ in range(80):
    o.zero_grad(); f(m(X), y).backward(); o.step(); sched.step(); lrs.append(o.param_groups[0]["lr"])
plt.plot(lrs); plt.title("Cosine annealing schedule"); plt.xlabel("step"); plt.ylabel("lr"); plt.show()

### Mini-exercise
Find a single SGD learning rate (no momentum, no schedule) that reaches loss < 0.02 within 60 steps on this problem. What is the largest rate that still converges?

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
for lr in [0.05, 0.1, 0.2, 0.3]:
    torch.manual_seed(0); m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr); f = nn.MSELoss()
    for _ in range(60):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step()
    print(f"lr={lr}: final loss {l.item():.4f}")

**Recap:** the learning rate is the most important hyperparameter; momentum and Adam smooth and adapt the updates; the loss curve diagnoses what went wrong.

---
Student materials for this week: the lab handout (`labs/week06.html`) and the curated references (`references/week06.html`) in the course site.